# Traffic Volume Prediction Using Machine Learning
## A Civil Engineering & Transportation Systems Case Study

**Objective:** Predict hourly traffic volume on Interstate 94 (I-94 Westbound) in the Minneapolis-St. Paul metropolitan area using historical traffic counts and OpenWeatherMap environmental features.

**Domain:** Transportation Engineering, Intelligent Transportation Systems (ITS), Urban Traffic Planning.

---

### 1. Import Required Libraries

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Plot style setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries successfully imported.")

### 2. Dataset Loading and Inspection
We load the Metro Interstate Traffic Volume dataset from the `../data/` directory.

In [ ]:
data_path = os.path.join("..", "data", "cleaned_traffic_data.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("..", "data", "Metro_Interstate_Traffic_Volume.csv")

print(f"Reading data from: {data_path}")
# If data exists, read it; otherwise notify
try:
    df = pd.read_csv(data_path)
    print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    display(df.head())
except Exception as e:
    print("Dataset not found locally yet. Run 'python scripts/prepare_data.py' first.")

### 3. Exploratory Data Analysis (EDA)
Civil engineers investigate diurnal (hourly) and seasonal (monthly) patterns in traffic flow.

In [ ]:
# If date_time column exists, parse it
if 'date_time' in df.columns:
    df['date_time'] = pd.to_datetime(df['date_time'])
    df['hour'] = df['date_time'].dt.hour
    df['day_of_week'] = df['date_time'].dt.dayofweek
    df['month'] = df['date_time'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Hourly Traffic Volume Profile
hourly_stats = df.groupby('hour')['traffic_volume'].mean()

plt.figure(figsize=(10, 5))
plt.plot(hourly_stats.index, hourly_stats.values, marker='o', color='#2c7a7b', linewidth=2.5)
plt.title('Average Hourly Traffic Volume (Vehicles / Hour)', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day (0 - 23)', fontsize=12)
plt.ylabel('Traffic Volume (veh/h)', fontsize=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 4. Feature Engineering: Cyclical Encoding
Time features like hour (0-23) wrap around circularly: hour 23 (11 PM) and hour 0 (midnight) are adjacent in physical reality, but numerically separated by 23 units. Sine and cosine transformations map them onto a continuous unit circle:

$$\text{hour}_{\sin} = \sin\left(\frac{2\pi \times \text{hour}}{24}\right), \quad \text{hour}_{\cos} = \cos\left(\frac{2\pi \times \text{hour}}{24}\right)$$
$$\text{month}_{\sin} = \sin\left(\frac{2\pi \times \text{month}}{12}\right), \quad \text{month}_{\cos} = \cos\left(\frac{2\pi \times \text{month}}{12}\right)$$
$$\text{dow}_{\sin} = \sin\left(\frac{2\pi \times \text{day\_of\_week}}{7}\right), \quad \text{dow}_{\cos} = \cos\left(\frac{2\pi \times \text{day\_of\_week}}{7}\right)$$

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

if 'is_holiday' not in df.columns:
    df['is_holiday'] = (df['holiday'] != 'None').astype(int)

print("Feature engineering complete. Sample features:")
display(df[['hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']].head())

### 5. Chronological Train-Test Split (80 / 20)
In time-series transportation data, we **never shuffle** the observations. Shuffling introduces **data leakage** from future conditions into past predictions. We use the first 80% for training and the remaining 20% chronologically for testing.

In [ ]:
feature_cols = [
    'temp', 'rain_1h', 'snow_1h', 'clouds_all',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday'
]
target_col = 'traffic_volume'

split_index = int(0.80 * len(df))
X = df[feature_cols]
y = df[target_col]

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training instances: {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Testing instances:  {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)")

### 6. Pipeline Construction & Model Training
We encapsulate `StandardScaler` and `LinearRegression` inside a `Pipeline` to prevent data leakage during scaling.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Fit model on training set
pipeline.fit(X_train, y_train)
print("Model pipeline fitted successfully.")

### 7. Model Evaluation
We evaluate Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the Coefficient of Determination ($R^2$).

In [ ]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

metrics_df = pd.DataFrame({
    'Metric': ['Mean Absolute Error (MAE)', 'Root Mean Squared Error (RMSE)', 'R-squared (R²) Score'],
    'Unit': ['veh/h', 'veh/h', 'dimensionless'],
    'Train Performance': [f"{train_mae:.2f}", f"{train_rmse:.2f}", f"{train_r2:.4f}"],
    'Test Performance': [f"{test_mae:.2f}", f"{test_rmse:.2f}", f"{test_r2:.4f}"]
})
display(metrics_df)

### 8. Residual Analysis
A residual is the difference between ground truth and predicted value: $e_i = y_i - \hat{y}_i$.

In [ ]:
residuals = y_test - y_test_pred

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# 1. Residual distribution
sns.histplot(residuals, kde=True, ax=ax[0], color='#2c7a7b')
ax[0].set_title('Residual Error Distribution (e = Actual - Predicted)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Error (veh/h)')
ax[0].axvline(0, color='red', linestyle='--')

# 2. Residuals vs Predicted
ax[1].scatter(y_test_pred[:500], residuals[:500], alpha=0.4, color='#1a365d')
ax[1].set_title('Residuals vs Predicted Traffic Volume', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Predicted Traffic Volume (veh/h)')
ax[1].set_ylabel('Residual (veh/h)')
ax[1].axhline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

### 9. Civil Engineering Conclusions
1. **Baseline Utility:** Multiple Linear Regression provides a fast, transparent baseline for macroscopic volume trends.
2. **Nonlinearity:** Because morning and evening peak hours create distinct non-linear surges, linear assumptions lead to moderate errors during transition hours.
3. **Practical Deployment:** In transportation engineering, this model output provides expected hourly demand for macroscopic road capacity planning, Level of Service (LOS) assessment, and seasonal maintenance scheduling.